# Thunderstorm Development Index Tutorial

This notebook demonstrates how to download HRRR model data, process key atmospheric variables, and prepare inputs for a Thunderstorm Development Index.

By the end of this notebook, you will be able to:
- Download HRRR data using Herbie
- Extract key meteorological variables
- Process and convert units
- Compute derived fields (e.g., convergence)
- Prepare inputs for a composite index

This workflow can be adapted for other forecasting or research applications.

## 1. Import Required Libraries

We begin by importing all necessary Python libraries for data access, processing, and visualization.

In [9]:
# Import statements
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import metpy.calc as mpcalc
from metpy.units import units
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as mcolors
from metpy.calc import lat_lon_grid_deltas
import os
from herbie import Herbie, FastHerbie
import dask

## 2. Define Data Configuration

We specify:
- Model type (HRRR)
- Forecast hours
- Storage location

In [10]:
#Date range for FastHerbie
DATES = pd.date_range(
    start="2026-04-20 00:00",
    periods=48,
    freq="1h",
)

# Create a range of forecast lead times
fxx = list(range(0, 24))

DATES

DatetimeIndex(['2026-04-20 00:00:00', '2026-04-20 01:00:00',
               '2026-04-20 02:00:00', '2026-04-20 03:00:00',
               '2026-04-20 04:00:00', '2026-04-20 05:00:00',
               '2026-04-20 06:00:00', '2026-04-20 07:00:00',
               '2026-04-20 08:00:00', '2026-04-20 09:00:00',
               '2026-04-20 10:00:00', '2026-04-20 11:00:00',
               '2026-04-20 12:00:00', '2026-04-20 13:00:00',
               '2026-04-20 14:00:00', '2026-04-20 15:00:00',
               '2026-04-20 16:00:00', '2026-04-20 17:00:00',
               '2026-04-20 18:00:00', '2026-04-20 19:00:00',
               '2026-04-20 20:00:00', '2026-04-20 21:00:00',
               '2026-04-20 22:00:00', '2026-04-20 23:00:00',
               '2026-04-21 00:00:00', '2026-04-21 01:00:00',
               '2026-04-21 02:00:00', '2026-04-21 03:00:00',
               '2026-04-21 04:00:00', '2026-04-21 05:00:00',
               '2026-04-21 06:00:00', '2026-04-21 07:00:00',
               '2026-04-

## 3. Download HRRR Model Data

We use Herbie to access HRRR data. Only specific variables are downloaded to reduce file size.
Just to mention there is an issue when trying to download all the data since the winds do not get combined with this step so the files have to be merged together to have all the data. 

In [20]:
# Fast Herbie Download
FH = FastHerbie(DATES, model="hrrr", fxx=fxx)
FH

Could not find 200/952 GRIB files.


In [21]:
# Fast Herbie objects
FH.objects, len(FH.objects) 

([▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F00 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F01 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F02 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F03 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F04 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F05 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F06 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F07 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F08 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F09 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F10 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr

In [22]:
# Run chosen for the time frame you decide for the index
run = "2026-04-15 00:00"

In [23]:
# Saved run as variables
H = FastHerbie([run], model="hrrr", fxx=np.arange(0,49,1).tolist(), save_dir='meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads', overwrite=True)

In [24]:
# Sort for variables with in saved run that you want to acquire
ss = "(:TMP:2 m|:CAPE:surface:|:TCDC:entire|:CIN:surface:|:DPT:2 m above)"
H.inventory(ss)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

,grib_message,start_byte,end_byte,range,reference_time,valid_time,variable,level,forecast_time,search_this,FILE
0,71,37436570,38658796.0,37436570-38658796,2026-04-15,2026-04-15,TMP,2 m above ground,anl,:TMP:2 m above ground:anl,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
1,74,41245489,42471688.0,41245489-42471688,2026-04-15,2026-04-15,DPT,2 m above ground,anl,:DPT:2 m above ground:anl,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
2,105,63803507,64409415.0,63803507-64409415,2026-04-15,2026-04-15,CAPE,surface,anl,:CAPE:surface:anl,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
3,106,64409416,64768530.0,64409416-64768530,2026-04-15,2026-04-15,CIN,surface,anl,:CIN:surface:anl,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
4,116,70925700,71934232.0,70925700-71934232,2026-04-15,2026-04-15,TCDC,entire atmosphere,anl,:TCDC:entire atmosphere:anl,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
...,...,...,...,...,...,...,...,...,...,...,...
240,71,46826780,48061955.0,46826780-48061955,2026-04-15,2026-04-17,TMP,2 m above ground,48 hour fcst,:TMP:2 m above ground:48 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
241,74,50621238,51848887.0,50621238-51848887,2026-04-15,2026-04-17,DPT,2 m above ground,48 hour fcst,:DPT:2 m above ground:48 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
242,108,77733767,78286761.0,77733767-78286761,2026-04-15,2026-04-17,CAPE,surface,48 hour fcst,:CAPE:surface:48 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
243,109,78286762,78999609.0,78286762-78999609,2026-04-15,2026-04-17,CIN,surface,48 hour fcst,:CIN:surface:48 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...


In [25]:
# Downloaded run
fp = H.download(ss)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

In [27]:
# Printed out data from run
fp

[PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d150bd9__hrrr.t00z.wrfsfcf03.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d1a0bd9__hrrr.t00z.wrfsfcf05.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8db80bd9__hrrr.t00z.wrfsfcf15.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d4b0bd9__hrrr.t00z.wrfsfcf04.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d890bd9__hrrr.t00z.wrfsfcf18.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d210bd9__hrrr.t00z.wrfsfcf02.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8da40bd9__hrrr.t00z.wrfsfcf11.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownlo

In [28]:
# Step to save data as a xarray file for later use
ds = xr.open_mfdataset(fp, combine='nested', concat_dim='valid_time')

Ignoring index file '/courses/meteo473/sp26/473_sp26_group4/Milestone 3/meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d150bd9__hrrr.t00z.wrfsfcf03.grib2.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group4/Milestone 3/meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d1a0bd9__hrrr.t00z.wrfsfcf05.grib2.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group4/Milestone 3/meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8db80bd9__hrrr.t00z.wrfsfcf15.grib2.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group4/Milestone 3/meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d4b0bd9__hrrr.t00z.wrfsfcf04.grib2.5b7b6.idx' older than GRIB file
Ignoring index file '/courses/meteo473/sp26/473_sp26_group4/Milestone 3/meteo473/sp26_groupwork/473_sp26_group4/

In [29]:
# xarray sorted by valued times to go from start to end
ds = ds.sortby('valid_time')
ds

<xarray.Dataset> Size: 2GB
Dimensions:            (valid_time: 49, y: 1059, x: 1799)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 392B 2026-04-15 ... 2026-0...
    time               datetime64[ns] 8B 2026-04-15
    step               (valid_time) timedelta64[ns] 392B 00:00:00 ... 2 days ...
    heightAboveGround  float64 8B 2.0
    latitude           (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude          (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    surface            float64 8B 0.0
    atmosphere         float64 8B 0.0
Dimensions without coordinates: y, x
Data variables:
    t2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    d2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    cape               (valid_time, y, x) float32 373MB dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    cin                (valid_time, y, x) float32 373MB dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    tcc                (valid_time, y, x) float32 373MB dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-30T18:26 GRIB to CDM+CF via cfgrib-0.9.1...

In [31]:
# Remaning the file based on your preference
filename = 'tutorial.nc'

In [32]:
# Saved Net CFD file that will pop up in the directory
ds.to_netcdf(filename)

## 4. Load Wind Data to Combine NetCDF files

Each variable from the previous step downloaded correcty, except the wind data which will be shown here and then merged into a single dataset.

In [33]:
# Fast Herbie Download
FH = FastHerbie(DATES, model="hrrr", fxx=fxx)
FH

Exception has occured : 'href'
Exception details : 'href'
Traceback (most recent call last):
  File "/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/fast.py", line 154, in __init__
    future.result()
    ~~~~~~~~~~~~~^^
  File "/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/concurrent/futures/_base.py", line 449, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/concurrent/futures/_base.py", line 401, in __get_result
    raise self._exception
  File "/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/concurrent/futures/thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
  File "/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py", line 309, in __init__
    self.idx, self.idx_source = self.find_idx()
                                ~~~~~~~~~~~~~^^
 

In [34]:
# Fast Herbie objects
FH.objects, len(FH.objects) 

([▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F00 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F01 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F02 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F03 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F04 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F05 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F06 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F07 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F08 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F09 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr-20 00:00 UTC F10 ┊ source=aws,
  ▌▌Herbie HRRR model sfc product initialized 2026-Apr

In [35]:
# Run chosen for the time frame you decide for the index
run = "2026-04-15 00:00"

In [36]:
# Saved run as variables
H = FastHerbie([run], model="hrrr", fxx=np.arange(0,49,1).tolist(), save_dir='meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads', overwrite=True)

In [37]:
# Sort for variables with in saved run that you want to acquire
ss = "(:UGRD:10 m above|:VGRD:10 m above)"
H.inventory(ss)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

,grib_message,start_byte,end_byte,range,reference_time,valid_time,variable,level,forecast_time,search_this,FILE
0,77,44464468,46846082.0,44464468-46846082,2026-04-15,2026-04-15 00:00:00,UGRD,10 m above ground,anl,:UGRD:10 m above ground:anl,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
1,78,46846083,48989554.0,46846083-48989554,2026-04-15,2026-04-15 00:00:00,VGRD,10 m above ground,anl,:VGRD:10 m above ground:anl,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
2,77,50392895,52774509.0,50392895-52774509,2026-04-15,2026-04-15 01:00:00,UGRD,10 m above ground,1 hour fcst,:UGRD:10 m above ground:1 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
3,78,52774510,55156124.0,52774510-55156124,2026-04-15,2026-04-15 01:00:00,VGRD,10 m above ground,1 hour fcst,:VGRD:10 m above ground:1 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
4,77,50925514,53307128.0,50925514-53307128,2026-04-15,2026-04-15 02:00:00,UGRD,10 m above ground,2 hour fcst,:UGRD:10 m above ground:2 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
...,...,...,...,...,...,...,...,...,...,...,...
93,78,56987213,59368827.0,56987213-59368827,2026-04-15,2026-04-16 22:00:00,VGRD,10 m above ground,46 hour fcst,:VGRD:10 m above ground:46 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
94,77,54710163,56853634.0,54710163-56853634,2026-04-15,2026-04-16 23:00:00,UGRD,10 m above ground,47 hour fcst,:UGRD:10 m above ground:47 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
95,78,56853635,59235249.0,56853635-59235249,2026-04-15,2026-04-16 23:00:00,VGRD,10 m above ground,47 hour fcst,:VGRD:10 m above ground:47 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...
96,77,53795434,55938905.0,53795434-55938905,2026-04-15,2026-04-17 00:00:00,UGRD,10 m above ground,48 hour fcst,:UGRD:10 m above ground:48 hour fcst,https://noaa-hrrr-bdp-pds.s3.amazonaws.com/hrr...


In [38]:
# Downloaded run
fp = H.download(ss)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

In [39]:
# Printed out data from run
fp

[PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d4b391b__hrrr.t00z.wrfsfcf04.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d15391b__hrrr.t00z.wrfsfcf03.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8dc5391b__hrrr.t00z.wrfsfcf08.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8dd4391b__hrrr.t00z.wrfsfcf10.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8da4391b__hrrr.t00z.wrfsfcf11.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d18391b__hrrr.t00z.wrfsfcf14.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownloads/hrrr/20260415/subset_8d9e391b__hrrr.t00z.wrfsfcf09.grib2'),
 PosixPath('meteo473/sp26_groupwork/473_sp26_group4/NewVariableDownlo

In [40]:
# Step to save data as a xarray file for later use
ds = xr.open_mfdataset(fp, combine='nested', concat_dim='valid_time')

/tmp/ipykernel_3423709/18422236.py:2: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds = xr.open_mfdataset(fp, combine='nested', concat_dim='valid_time')


In [41]:
# xarray sorted by valued times to go from start to end
ds = ds.sortby('valid_time')
ds

<xarray.Dataset> Size: 777MB
Dimensions:            (valid_time: 49, y: 1059, x: 1799)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 392B 2026-04-15 ... 2026-0...
    time               datetime64[ns] 8B 2026-04-15
    step               (valid_time) timedelta64[ns] 392B 00:00:00 ... 2 days ...
    heightAboveGround  float64 8B 10.0
    latitude           (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude          (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    u10                (valid_time, y, x) float32 373MB dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    v10                (valid_time, y, x) float32 373MB dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-30T18:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [59]:
# Renaming the file based on your preference
filename = 'tutorialwinds.nc'

In [43]:
# Saved Net CFD file that will pop up in the directory
ds.to_netcdf(filename)

After wind data is downloaded now its time to combine the NetCDF files to use fro the index 

In [45]:
# Save the files with a variable of your choice
data = 'tutorial.nc'
winds = 'tutorialwinds.nc'

In [50]:
# Save each of the xarray files for the two data sets and labled them as difference varaibles
ds1 = xr.open_mfdataset(data)
ds2 = xr.open_mfdataset(winds)

sh: 1: getfattr: not found
sh: 1: getfattr: not found


In [54]:
# Printed out each of the xarray files to see them
ds1

<xarray.Dataset> Size: 2GB
Dimensions:            (valid_time: 49, y: 1059, x: 1799)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 392B 2026-04-15 ... 2026-0...
    time               datetime64[ns] 8B ...
    step               (valid_time) timedelta64[ns] 392B dask.array<chunksize=(49,), meta=np.ndarray>
    heightAboveGround  float64 8B ...
    latitude           (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude          (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    surface            float64 8B ...
    atmosphere         float64 8B ...
Dimensions without coordinates: y, x
Data variables:
    t2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    d2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    cape               (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    cin                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    tcc                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-30T18:26 GRIB to CDM+CF via cfgrib-0.9.1...

In [55]:
# Printed out each of the xarray files to see them
ds2

<xarray.Dataset> Size: 777MB
Dimensions:            (valid_time: 49, y: 1059, x: 1799)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 392B 2026-04-15 ... 2026-0...
    time               datetime64[ns] 8B ...
    step               (valid_time) timedelta64[ns] 392B dask.array<chunksize=(49,), meta=np.ndarray>
    heightAboveGround  float64 8B ...
    latitude           (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude          (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    u10                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    v10                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-30T18:35 GRIB to CDM+CF via cfgrib-0.9.1...

In [56]:
# Merged function of the two xarray dataset to combine into NetCDF
ds_merge_tutorial = xr.merge([ds1,ds2], compat = 'override')

In [58]:
# Printed out merged xarray dataset - notice in the data varaibles section it now has all the varaible listed you want for the index
ds_merge_tutorial

<xarray.Dataset> Size: 3GB
Dimensions:            (valid_time: 49, y: 1059, x: 1799)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 392B 2026-04-15 ... 2026-0...
    time               datetime64[ns] 8B ...
    step               (valid_time) timedelta64[ns] 392B dask.array<chunksize=(49,), meta=np.ndarray>
    heightAboveGround  float64 8B ...
    latitude           (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude          (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    surface            float64 8B ...
    atmosphere         float64 8B ...
Dimensions without coordinates: y, x
Data variables:
    t2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    d2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    cape               (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    cin                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    tcc                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    u10                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    v10                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-30T18:26 GRIB to CDM+CF via cfgrib-0.9.1...

In [61]:
# Renaming the file based on your preference
filename = ('Tutorial')

In [62]:
# Saved Net CFD file that will pop up in the directory
ds.to_netcdf(filename)

## 7. Define Index Component Functions and Map Functions

At this stage, model data are prepared for index calculation by selecting the desired forecast time and converting variables into consistent, interpretable units (e.g., temperature and dewpoint to °F, cloud cover to %).

Each variable is then transformed into a normalized 0–100 contribution using physically based functions. These transformations reflect how each parameter influences thunderstorm development (e.g., increasing CAPE enhances instability, while stronger CIN suppresses convection).

Normalizing all variables to a common scale allows them to be directly compared and combined into a single weighted index representing overall thunderstorm potential.

After the index is computed, plots are automatically generated for each forecast hour (e.g., 0–24 hours) and saved to a designated folder, allowing quick visualization and review of the full time evolution of the index.

In [105]:
ds_merge_tutorial

<xarray.Dataset> Size: 3GB
Dimensions:            (valid_time: 49, y: 1059, x: 1799)
Coordinates:
  * valid_time         (valid_time) datetime64[ns] 392B 2026-04-15 ... 2026-0...
    time               datetime64[ns] 8B ...
    step               (valid_time) timedelta64[ns] 392B dask.array<chunksize=(49,), meta=np.ndarray>
    heightAboveGround  float64 8B ...
    latitude           (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude          (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    surface            float64 8B ...
    atmosphere         float64 8B ...
Dimensions without coordinates: y, x
Data variables:
    t2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    d2m                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    cape               (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    cin                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    tcc                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    u10                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
    v10                (valid_time, y, x) float32 373MB dask.array<chunksize=(49, 1059, 1799), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-30T18:26 GRIB to CDM+CF via cfgrib-0.9.1...

In [106]:
# Choose one time index (you can change this later)
fcst = ds_merge_tutorial.isel(valid_time=48)

# Extract initialization and valid times
init_time = pd.to_datetime(ds_merge_tutorial.time.values)
valid_time = pd.to_datetime(fcst.valid_time.values)

# Format times for plot titles
init_str = init_time.strftime('%HZ %b %d %Y')
valid_str = valid_time.strftime('%HZ %b %d %Y')

# Print out initial and valid times
print("Init Time:", init_str)
print("Valid Time:", valid_str)

Init Time: 00Z Apr 15 2026
Valid Time: 00Z Apr 17 2026


In [107]:
# Extract 10-meter wind components and compute wind speed magnitude
u10 = fcst['u10']
v10 = fcst['v10']
wind_speed = np.sqrt(u10**2 + v10**2)

In [108]:
# Convert temperatures to Fahrenheit
temp_f = (fcst['t2m'] - 273.15) * 9/5 + 32
dewpt_f = (fcst['d2m'] - 273.15) * 9/5 + 32

cape = fcst['cape']
cin = fcst['cin']
cloud = fcst['tcc'] * 100

# Compute dx and dy from lat/lon
dx, dy = lat_lon_grid_deltas(fcst['longitude'].values, fcst['latitude'].values)

# Compute convergence (negative divergence)
convergence = -mpcalc.divergence(fcst['u10'], fcst['v10'], dx=dx, dy=dy)

# Define function for Threat Index
def threat_index(temp_f, dewpt_f, convergence, cape, cin, cloud):

    # Temperature (best 60–90°F)
    temp = np.clip(100 * np.exp(-((temp_f - 75)**2) / 200), 0, 100)

    # Dewpoint (logistic growth)
    dew_point = 100 / (1 + np.exp(-0.2 * (dewpt_f - 60)))

    # CAPE (exponential growth)
    cape_index = 100 * (1 - np.exp(-cape / 1000))

    # CIN (inverse exponential decay)
    cin_index = 100 * np.exp(-np.abs(cin) / 100)

    # Cloud cover (logistic)
    cloud_cover = 100 / (1 + np.exp(-0.1 * (cloud - 50)))

    # Convergence (scaled)
    convergent_winds = np.clip(convergence * 1e5, 0, 100)

     # Weighted Index Calculation
    index = (0.15 * temp + 0.25 * dew_point + 0.05 * convergent_winds +
             0.30 * cape_index + 0.20 * cin_index + 0.05 * cloud_cover)

    return index

# Printed out index in xarray
index = threat_index(temp_f, dewpt_f, convergence, cape, cin, cloud)
index

<xarray.DataArray (y: 1059, x: 1799)> Size: 15MB
dask.array<add, shape=(1059, 1799), dtype=float64, chunksize=(1059, 1799), chunktype=numpy.ndarray>
Coordinates:
    time               datetime64[ns] 8B 2026-04-15
    step               timedelta64[ns] 8B dask.array<chunksize=(), meta=np.ndarray>
    heightAboveGround  float64 8B 2.0
    latitude           (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude          (y, x) float64 15MB dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    valid_time         datetime64[ns] 8B 2026-04-17
    surface            float64 8B 0.0
    atmosphere         float64 8B 0.0
Dimensions without coordinates: y, x
Attributes: (12/26)
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      1905141
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    GRIB_gridType:                            lambert
    GRIB_uvRelativeToGrid:                    1
    ...                                       ...
    GRIB_latitudeOfFirstGridPointInDegrees:   21.138123
    GRIB_latitudeOfSouthernPoleInDegrees:     0.0
    GRIB_longitudeOfFirstGridPointInDegrees:  237.280472
    GRIB_longitudeOfSouthernPoleInDegrees:    0.0
    GRIB_missingValue:                        3.4028234663852886e+38
    standard_name:                            unknown

In [109]:
# Automated plots in folder to see all the timestamps for the 48 hours

# Ensure output directory exists
outdir = os.path.join(os.getcwd(), "Tutorial_Plots")
os.makedirs(outdir, exist_ok=True)

# CONUS Map Function
def conus_map():
    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([-125, -65, 24, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES)
    return fig, ax

# Threat index function
def threat_index(temp_f, dewpt_f, convergence, cape, cin, cloud):
    temp = np.clip(100 * np.exp(-((temp_f - 75)**2) / 200), 0, 100)
    dew_point = 100 / (1 + np.exp(-0.2 * (dewpt_f - 60)))
    cape_index = 100 * (1 - np.exp(-cape / 1000))
    cin_index = 100 * np.exp(-np.abs(cin) / 100)
    cloud_cover = 100 / (1 + np.exp(-0.1 * (cloud - 50)))
    convergent_winds = np.clip(convergence * 1e5, 0, 100)

    index = (0.15 * temp + 0.25 * dew_point + 0.05 * convergent_winds +
             0.30 * cape_index + 0.20 * cin_index + 0.05 * cloud_cover)
    return index

# Loop over all times in the dataset
for i in range(len(ds.valid_time)):

    fcst = ds_merge_tutorial.isel(valid_time=i)

    # Convert times
    init_time = pd.to_datetime(ds_merge_tutorial.time.values)
    valid_time = pd.to_datetime(fcst.valid_time.values)
    init_str = init_time.strftime('%HZ %b %d %Y')
    valid_str = valid_time.strftime('%HZ %b %d %Y')

    # Compute fields
    temp_f = (fcst['t2m'] - 273.15) * 9/5 + 32
    dewpt_f = (fcst['d2m'] - 273.15) * 9/5 + 32
    cape = fcst['cape']
    cin = fcst['cin']
    cloud = fcst['tcc'] * 100

    dx, dy = lat_lon_grid_deltas(fcst['longitude'].values, fcst['latitude'].values)
    convergence = -mpcalc.divergence(fcst['u10'], fcst['v10'], dx=dx, dy=dy)

    # Compute threat index
    index = threat_index(temp_f, dewpt_f, convergence, cape, cin, cloud)

    # Plotted conus map
    fig, ax = conus_map()

    bounds = np.linspace(0, 100, 101)
    norm = mcolors.BoundaryNorm(bounds, ncolors=256)

    c = ax.contourf(
        fcst['longitude'],
        fcst['latitude'],
        index,
        cmap='RdYlGn_r',
        levels=bounds,
        norm=norm,
        shading='auto'
    )

    cb = plt.colorbar(
        c,
        ax=ax,
        orientation='horizontal',
        pad=0.02,
        ticks=[0, 20, 40, 60, 80, 100]
    )

    plt.title("Thunderstorm Development Probability Index (0–100)", loc='center', fontweight='bold')
    plt.title(f"Init: {init_str}", loc='left')
    plt.title(f"Valid: {valid_str}", loc='right')

    plt.text(
        0.01, 0.01,
        "Parameters: Temp, Dewpoint, CAPE, CIN, Convergence, Cloud Cover",
        transform=ax.transAxes,
        fontsize=9,
        color='white',
        bbox=dict(facecolor='black', alpha=0.4, pad=3)
    )

    # Save files as pngs and save them in a folder of all 48 hours for incident day
    fname = f"threat_{i:03d}.png"
    plt.savefig(os.path.join(outdir, fname), dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved {fname}")


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/cartopy/mpl/geoaxes.py:1631: UserWarning: The following kwargs were not used by contour: 'shading'
  result = super().contourf(*args, **kwargs)


Saved threat_000.png


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/cartopy/mpl/geoaxes.py:1631: UserWarning: The following kwargs were not used by contour: 'shading'
  result = super().contourf(*args, **kwargs)


Saved threat_001.png


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/cartopy/mpl/geoaxes.py:1631: UserWarning: The following kwargs were not used by contour: 'shading'
  result = super().contourf(*args, **kwargs)


...

Saved threat_046.png


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/cartopy/mpl/geoaxes.py:1631: UserWarning: The following kwargs were not used by contour: 'shading'
  result = super().contourf(*args, **kwargs)


Saved threat_047.png


/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/cartopy/mpl/geoaxes.py:1631: UserWarning: The following kwargs were not used by contour: 'shading'
  result = super().contourf(*args, **kwargs)


Saved threat_048.png
